In [22]:
import pandas as pd
from google.colab import files

# Upload all required CSV files
uploaded = files.upload()

Saving admission.csv to admission (2).csv
Saving bed.csv to bed (2).csv
Saving billing.csv to billing (2).csv
Saving department.csv to department (2).csv
Saving disease.csv to disease (2).csv
Saving doctor.csv to doctor (2).csv
Saving employee.csv to employee (2).csv
Saving patient.csv to patient (2).csv
Saving staff_assignment.csv to staff_assignment (2).csv
Saving ward.csv to ward (2).csv


In [23]:
patients = pd.read_csv("patient.csv")
admissions = pd.read_csv("admission.csv")
departments = pd.read_csv("department.csv")
wards = pd.read_csv("ward.csv")
beds = pd.read_csv("bed.csv")
billing = pd.read_csv("billing.csv")
staff_assignment = pd.read_csv("staff_assignment.csv")
employee = pd.read_csv("employee.csv")
patient_insurance = pd.read_csv("insurance_provider.csv")
disease = pd.read_csv("disease.csv")
patient_diagnostic = pd.read_csv("diagnostic_test.csv")

In [24]:
tables = {
    "Patients": patients,
    "Admissions": admissions,
    "Departments": departments,
    "Wards": wards,
    "Beds": beds,
    "Billing": billing,
    "Staff Assignment": staff_assignment,
    "Employee": employee,
    "Disease": disease,
}

for name, df in tables.items():
    print(name)
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

Patients
Rows: 30000
Columns: ['patient_id', 'gender', 'date_of_birth', 'blood_group', 'city', 'contact_number']
Admissions
Rows: 45000
Columns: ['admission_id', 'admission_date', 'discharge_date', 'admission_type', 'admission_status', 'patient_id', 'department_id', 'ward_id', 'bed_id', 'disease_id']
Departments
Rows: 11
Columns: ['department_id', 'department_name', 'department_type', 'floor_number', 'status']
Wards
Rows: 27
Columns: ['ward_id', 'ward_name', 'ward_type', 'total_beds', 'department_id']
Beds
Rows: 415
Columns: ['bed_id', 'bed_number', 'bed_status', 'ward_id']
Billing
Rows: 45000
Columns: ['bill_id', 'bill_date', 'total_amount', 'insurance_covered_amount', 'patient_payable_amount', 'payment_status', 'payment_mode', 'admission_id']
Staff Assignment
Rows: 207
Columns: ['assignment_id', 'employee_id', 'ward_id', 'shift']
Employee
Rows: 500
Columns: ['employee_id', 'employee_name', 'gender', 'role', 'employment_type', 'date_of_joining', 'department_id']
Disease
Rows: 20
Colum

In [25]:

raw_master = admissions.copy()
raw_master = raw_master.merge(
    patients,
    on="patient_id",
    how="left"
)
raw_master = raw_master.merge(
    departments,
    on="department_id",
    how="left",
    suffixes=("", "_department")
)
raw_master = raw_master.merge(
    wards,
    on="ward_id",
    how="left",
    suffixes=("", "_ward")
)
raw_master = raw_master.merge(
    beds,
    on="bed_id",
    how="left",
    suffixes=("", "_bed")
)
raw_master = raw_master.merge(
    disease,
    on="disease_id",
    how="left",
    suffixes=("", "_disease")
)
billing_summary = (
    billing
    .groupby("admission_id")
    .agg(
        Total_Billing=("total_amount", "sum"),
        Insurance_Covered_Amount=(
            "insurance_covered_amount",
            "sum"
        ),
        Patient_Payable_Amount=(
            "patient_payable_amount",
            "sum"
        )
    )
    .reset_index()
)

raw_master = raw_master.merge(
    billing_summary,
    on="admission_id",
    how="left"
)

staff_summary = (
    staff_assignment
    .groupby("ward_id")
    .agg(
        Staff_Count=(
            "employee_id",
            "nunique"
        ),
        Staff_Assignment_Count=(
            "assignment_id",
            "nunique"
        )
    )
    .reset_index()
)

raw_master = raw_master.merge(
    staff_summary,
    on="ward_id",
    how="left"
)


employee_summary = (
    employee
    .groupby("department_id")
    .agg(
        Employee_Count=(
            "employee_id",
            "nunique"
        )
    )
    .reset_index()
)

raw_master = raw_master.merge(
    employee_summary,
    on="department_id",
    how="left"
)



In [27]:

print("RAW DATA INTEGRATION COMPLETED")
print("Total Rows:", len(raw_master))
print(
    "Total Columns:",
    len(raw_master.columns)
)



RAW DATA INTEGRATION COMPLETED
Total Rows: 45000
Total Columns: 34


In [30]:
file_name = "hospital_raw_integrated.xlsx"

raw_master.to_excel(
    file_name,
    index=False
)

print(
    "\n✅ File saved successfully:",
    file_name
)
files.download(file_name)


✅ File saved successfully: hospital_raw_integrated.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Cleaning DataSet


In [189]:
import pandas as pd
import numpy as np
df = pd.read_excel("/content/hospital_dirty_dataset(Infosys).xlsx")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
display(df.head())

Rows: 45900
Columns: 34


,admission_id,admission_date,discharge_date,admission_type,admission_status,patient_id,department_id,ward_id,bed_id,disease_id,...,bed_status,ward_id_bed,disease_name,disease_category,Total_Billing,Insurance_Covered_Amount,Patient_Payable_Amount,Staff_Count,Staff_Assignment_Count,Employee_Count
0,1,2020-02-25,2020-02-27,Emergency,Discharged,166,2,6,76,10,...,Available,6,Anemia,Hematological,68483,61634.7,6848.3,7,7,54
1,2,2022-02-22,2022-03-04,Elective,Discharged,8622,5,21,302,11,...,Available,21,Fracture Femur,Orthopedic,70917,35458.5,35458.5,7,7,40
2,3,2021-02-03,2021-02-09,Elective,Discharged,23976,1,2,11,9,...,Available,2,Chronic Obstructive Pulmonary Disease,Respiratory,28137,25323.3,2813.7,11,11,51
3,4,2021-12-31,2022-01-05,Elective,Discharged,16635,2,10,128,1,...,Available,10,Acute Myocardial Infarction,Cardiac,80665,64532.0,16133.0,6,6,54
4,5,2022-07-02,2022-07-07,Elective,Discharged,10654,3,11,157,7,...,Available,11,Hypertension,Cardiac,54920,27460.0,27460.0,5,5,43


In [190]:
# Check complete duplicate records

duplicate_count = df.duplicated().sum()

print("Duplicate records BEFORE cleaning:", duplicate_count)

if duplicate_count > 0:
    display(
        df[df.duplicated(keep=False)].head(20)
    )
else:
    print("No duplicate records found.")

Duplicate records BEFORE cleaning: 518


,admission_id,admission_date,discharge_date,admission_type,admission_status,patient_id,department_id,ward_id,bed_id,disease_id,...,bed_status,ward_id_bed,disease_name,disease_category,Total_Billing,Insurance_Covered_Amount,Patient_Payable_Amount,Staff_Count,Staff_Assignment_Count,Employee_Count
216,217,2024-07-03,2024-07-08,Elective,Discharged,7175,3,11,157,2,...,Available,11,Stroke,Neurological,7467,5973.6,1493.4,5,5,43
354,355,2025-05-15,2025-05-21,Elective,Discharged,23411,4,16,236,9,...,Available,16,Chronic Obstructive Pulmonary Disease,Respiratory,38212,19106.0,19106.0,7,7,33
646,647,2024-09-08,2024-09-12,Elective,Discharged,24495,4,18,271,5,...,Available,18,Acute Respiratory Distress,Respiratory,32592,26073.6,6518.4,5,5,33
884,885,2025-03-29,2025-04-06,Elective,Discharged,2607,1,4,48,18,...,Available,4,Viral Fever,Infectious,17646,14116.8,3529.2,5,5,51
1107,1108,2021-12-20,2021-12-27,Elective,Discharged,6460,2,10,131,7,...,Available,10,Hypertension,Cardiac,53564,42851.2,10712.8,6,6,54
1114,1115,2024-11-09,2024-11-18,Elective,Discharged,4372,3,13,183,20,...,Available,13,Tuberculosis,Infectious,5391,4851.9,539.1,10,10,43
1121,1122,2020-04-11,2020-04-20,Elective,Discharged,450,3,12,170,18,...,Available,12,Viral Fever,Infectious,32877,29589.3,3287.7,12,12,43
1232,1233,2025-08-16,2025-08-17,Emergency,Discharged,29338,2,10,136,16,...,Available,10,Neonatal Jaundice,Pediatric,10759,9683.1,1075.9,6,6,54
1377,1378,2025-03-28,2025-03-30,Emergency,Discharged,21976,4,17,256,4,...,Available,17,Sepsis,Infectious,42500,34000.0,8500.0,11,11,33
1603,1604,2025-06-23,2025-06-24,Emergency,Discharged,14522,1,4,46,7,...,Available,4,Hypertension,Cardiac,77690,38845.0,38845.0,5,5,51


In [192]:
if duplicate_count > 0:
    df = df.drop_duplicates().copy()
    print("Duplicate records removed.")
else:
    print("No duplicates to remove.")

Duplicate records removed.


In [195]:
remaining_duplicates = df.duplicated().sum()

print(
    "Duplicate records AFTER cleaning:",
    remaining_duplicates
)

if remaining_duplicates == 0:
    print(" Duplicate cleaning completed successfully.")

Duplicate records AFTER cleaning: 0
 Duplicate cleaning completed successfully.


In [164]:
#Column-wise missing values
missing_count = df.isnull().sum()

print(" MISSING VALUES BEFORE CLEANING")

display(
    missing_count[missing_count > 0]
)

 MISSING VALUES BEFORE CLEANING


,0
gender,459
date_of_birth,459
blood_group,459
city,459
contact_number,459


In [165]:
missing_percentage = (
    df.isnull().sum() / len(df)
) * 100

print("===== MISSING PERCENTAGE BEFORE CLEANING =====")

display(
    missing_percentage[missing_percentage > 0]
)

===== MISSING PERCENTAGE BEFORE CLEANING =====


,0
gender,1.011414
date_of_birth,1.011414
blood_group,1.011414
city,1.011414
contact_number,1.011414


In [166]:
# Overall missing percentage
total_missing = df.isnull().sum().sum()
total_cells = df.shape[0] * df.shape[1]

overall_missing_percentage = (
    total_missing / total_cells
) * 100

print("Total Missing Values:", total_missing)
print("Total Cells:", total_cells)
print(
    "Overall Missing Percentage:",
    round(overall_missing_percentage, 2),
    "%"
)

Total Missing Values: 2295
Total Cells: 1542988
Overall Missing Percentage: 0.15 %


In [167]:
# Handling
numeric_columns = df.select_dtypes(
    include=np.number
).columns

text_columns = df.select_dtypes(
    include="object"
).columns
# Numeric columns → median
for col in numeric_columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(
            df[col].median()
        )
# Text columns → mode
for col in text_columns:
    if df[col].isnull().sum() > 0:
        mode_value = df[col].mode()

        if len(mode_value) > 0:
            df[col] = df[col].fillna(
                mode_value[0]
            )

In [168]:
remaining_missing = df.isnull().sum().sum()
print(
    "Total missing values AFTER cleaning:",
    remaining_missing
)

if remaining_missing == 0:
    print(" Missing-value cleaning completed successfully.")
else:
    print(" Missing values still remain.")

    display(
        df.isnull().sum()[
            df.isnull().sum() > 0
        ]
    )

Total missing values AFTER cleaning: 0
 Missing-value cleaning completed successfully.


In [133]:
# Check spaces BEFORE standardization
print("SPACE CHECK BEFORE CLEANING")
space_count = 0
for col in text_columns:
    count = (
        df[col].notna()
        & (df[col] != df[col].str.strip())
    ).sum()
    if count > 0:
        print(
            col,
            ":",
            count,
            "records with leading/trailing spaces"
        )

        space_count += count

print(
    "\nTotal records containing spaces:",
    space_count
)

SPACE CHECK BEFORE CLEANING
admission_type : 219 records with leading/trailing spaces
admission_status : 223 records with leading/trailing spaces
gender : 267 records with leading/trailing spaces
department_name : 573 records with leading/trailing spaces
department_type : 244 records with leading/trailing spaces
ward_type : 235 records with leading/trailing spaces
bed_status : 245 records with leading/trailing spaces
disease_category : 250 records with leading/trailing spaces

Total records containing spaces: 2256


In [169]:
# Removing
for col in text_columns:
    df[col] = df[col].str.strip()

In [170]:
remaining_spaces = 0
for col in text_columns:
    remaining_spaces += (
        df[col].notna()
        & (df[col] != df[col].str.strip())
    ).sum()

print(
    "Remaining leading/trailing spaces:",
    remaining_spaces
)

if remaining_spaces == 0:
    print("Space cleaning completed.")

Remaining leading/trailing spaces: 0
Space cleaning completed.


In [171]:
# Standardize Department Names
print("DEPARTMENT NAMES BEFORE STANDARDIZATION")

display(
    df["department_name"]
    .value_counts()
)

DEPARTMENT NAMES BEFORE STANDARDIZATION


,count
department_name,
Surgery,9838
Emergency,8511
Pediatrics,8191
Internal Medicine,7479
Orthopedics,5735
ICU,3960
SURGERY,133
surgery,128
sURGERY,123


In [172]:
df["department_name"] = (
    df["department_name"]
    .astype(str)
    .str.strip()
    .str.title()
)

In [173]:
print("DEPARTMENT NAMES AFTER STANDARDIZATION")
display(
    df["department_name"]
    .value_counts()
)

DEPARTMENT NAMES AFTER STANDARDIZATION


,count
department_name,
Surgery,10222
Emergency,8850
Pediatrics,8510
Internal Medicine,7769
Orthopedics,5962
Icu,4069


In [174]:
# Data Types
print("DATA TYPES BEFORE CORRECTION")

display(df.dtypes)

DATA TYPES BEFORE CORRECTION


,0
admission_id,int64
admission_date,object
discharge_date,object
admission_type,object
admission_status,object
patient_id,int64
department_id,int64
ward_id,int64
bed_id,int64
disease_id,int64


In [178]:
# changing data types
date_columns = [
    "admission_date",
    "discharge_date",
    "date_of_birth"
]

for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(
            df[col],
            errors="coerce"
        )
numeric_cols = [
    "floor_number",
    "total_beds",
    "Total_Billing",
    "Insurance_Covered_Amount",
    "Patient_Payable_Amount",
    "Staff_Count",
    "Staff_Assignment_Count",
    "Employee_Count"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [179]:
print("DATA TYPES AFTER CORRECTION")
display(df.dtypes)

DATA TYPES AFTER CORRECTION


,0
admission_id,int64
admission_date,datetime64[ns]
discharge_date,datetime64[ns]
admission_type,object
admission_status,object
patient_id,int64
department_id,int64
ward_id,int64
bed_id,int64
disease_id,int64


In [180]:
if "blood_group" in df.columns:
    df["blood_group"] = (
        df["blood_group"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

In [181]:
# Extracting month and year
df["Admission_Month"] = (
    df["admission_date"]
    .dt.month_name()
)
df["Admission_Year"] = (
    df["admission_date"]
    .dt.year
)

In [182]:
# Normalizing Billing Vlaues
billing_columns = [
    "Total_Billing",
    "Insurance_Covered_Amount",
    "Patient_Payable_Amount"
]
for col in billing_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )
        df[col] = df[col].round(2)

In [183]:
# Checking negative values
for col in billing_columns:
    if col in df.columns:
        print(
            col,
            "negative values:",
            (df[col] < 0).sum()
        )

Total_Billing negative values: 0
Insurance_Covered_Amount negative values: 0
Patient_Payable_Amount negative values: 0


In [184]:
df["date_of_birth"] = pd.to_datetime(
    df["date_of_birth"],
    errors="coerce"
)
# Calculate Age
today = pd.Timestamp.today()

df["Age"] = (
    today.year - df["date_of_birth"].dt.year
    - (
        (today.month < df["date_of_birth"].dt.month)
        |
        (
            (today.month == df["date_of_birth"].dt.month)
            &
            (today.day < df["date_of_birth"].dt.day)
        )
    )
)

# Convert to nullable integer
df["Age"] = df["Age"].astype("Int64")

# Display calculated Age
display(
    df[
        ["date_of_birth", "Age"]
    ].head(10)
)

,date_of_birth,Age
0,1954-11-02,71
1,1988-02-25,38
2,1944-03-02,82
3,2017-08-19,8
4,2015-05-06,11
5,1960-01-21,66
6,1952-11-17,73
7,2008-05-12,18
8,1973-08-29,52
9,1960-10-08,65


In [148]:
# Dividing the age Group
df["Age Group"] = pd.cut(
    df["Age"],
    bins=[0, 12, 18, 35, 60, 120],
    labels=[
        "Child",
        "Teenager",
        "Young Adult",
        "Adult",
        "Senior"
    ],
    include_lowest=True
)
display(
    df[["Age", "Age Group"]].head(10)
)

,Age,Age Group
0,71,Senior
1,38,Adult
2,82,Senior
3,8,Child
4,11,Child
5,66,Senior
6,73,Senior
7,18,Teenager
8,52,Adult
9,65,Senior


In [149]:
# Creating Billing Category
df["Billing Category"] = pd.cut(
    df["Total_Billing"],
    bins=[0, 10000, 25000, 50000, float("inf")],
    labels=[
        "Low",
        "Medium",
        "High",
        "Very High"
    ],
    include_lowest=True
)

display(
    df[
        ["Total_Billing", "Billing Category"]
    ].head(10)
)

,Total_Billing,Billing Category
0,68483,Very High
1,70917,Very High
2,28137,High
3,80665,Very High
4,54920,Very High
5,71521,Very High
6,13951,Medium
7,46293,High
8,21830,Medium
9,18606,Medium


In [150]:
print("Total Patients:",
      df["patient_id"].nunique())
print("Total Admissions:",
      df["admission_id"].nunique())
print("Total Billing Amount:",
      round(df["Total_Billing"].sum(), 2))
print("Average Billing Amount:",
      round(df["Total_Billing"].mean(), 2))
print("Total Departments:",
      df["department_id"].nunique())
print("Total Wards:",
      df["ward_id"].nunique())
print("Total Beds:",
      df["bed_id"].nunique())
print("Total Diseases:",
      df["disease_id"].nunique())

Total Patients: 23275
Total Admissions: 45000
Total Billing Amount: 1698225707
Average Billing Amount: 37420.69
Total Departments: 6
Total Wards: 27
Total Beds: 145
Total Diseases: 20


In [151]:
# Check final dataset
print("Final dataset shape:", df.shape)
# Save the cleaned and transformed dataset
file_name = "hospital_cleaned_transformed_dataset.csv"
df.to_csv(file_name, index=False)
print("Dataset saved successfully as:", file_name)

Final dataset shape: (45382, 39)
Dataset saved successfully as: hospital_cleaned_transformed_dataset.csv


In [197]:
from google.colab import files

files.download("hospital_cleaned_transformed_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>